# Coordinate Descent and Convergence Rate Taxonomy

| Difficulty | █████░░░░░ 5/10 |
| :--- | :--- |
| Prerequisites | 04.10 |
| Optional | 04.05 |
| Time | ~70 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/04_Optimization/11_coordinate_descent_and_convergence_rates.ipynb)

---

## 🎯 Learning Objective

Understand coordinate descent as a scalable alternative to full-gradient methods for separable
regularizers, and place every first-order optimizer encountered so far into a unified convergence
rate taxonomy — from the slowest (subgradient $O(1/\sqrt{T})$) to the fastest (Newton's
quadratic convergence) — culminating in Nesterov's lower bound that proves certain rates are
unimprovable.

---

## 📐 Theory

### 1. Coordinate descent: one variable at a time

Full gradient descent updates every coordinate of $\mathbf{w} \in \mathbb{R}^d$ simultaneously
using the full gradient $\nabla f(\mathbf{w})$. **Coordinate descent (CD)** takes the opposite
approach: at each step, pick a single coordinate index $j$ and solve the one-dimensional
subproblem *exactly* (or approximately), keeping all other coordinates fixed:

$$w_j^{(t+1)} = \arg\min_{w_j}\; f\bigl(w_1^{(t+1)},\ldots, w_{j-1}^{(t+1)},\, w_j,\, w_{j+1}^{(t)},\ldots, w_d^{(t)}\bigr)$$

The key insight is that for **separable regularizers** like $g(\mathbf{w}) = \lambda\|\mathbf{w}\|_1
= \lambda\sum_j |w_j|$, the full $d$-dimensional non-smooth optimization problem decomposes into
$d$ independent one-dimensional problems, each of which often has a **closed-form solution**.
This sidesteps the need for proximal operators entirely — the non-smoothness is handled
analytically, one coordinate at a time.

Why prefer CD over full gradient methods? When $d$ is very large and the data matrix $X$ is
sparse, updating a single coordinate costs $O(\text{nnz}(X_j))$ — the number of nonzero entries
in column $j$ — rather than the $O(nd)$ cost of a full gradient. For problems where $d \gg n$
(common in genomics, NLP bag-of-words, recommender systems), this per-coordinate cheapness
makes CD dramatically faster in wall-clock time despite needing more "iterations" in the
coordinate sense.

### 2. Coordinate descent for Lasso: closed-form soft-thresholding update

Consider the Lasso objective from `04.10`:

$$\min_{\mathbf{w}}\; \frac{1}{2}\|X\mathbf{w} - \mathbf{y}\|^2 + \lambda\|\mathbf{w}\|_1$$

To derive the coordinate update for $w_j$, write the residual as
$\mathbf{r}_{-j} = \mathbf{y} - X_{-j}\mathbf{w}_{-j}$ (the residual with coordinate $j$
removed), so the objective as a function of $w_j$ alone becomes:

$$\frac{1}{2}\|\mathbf{r}_{-j} - X_j w_j\|^2 + \lambda|w_j|
= \frac{\|X_j\|^2}{2}\left(w_j - \frac{X_j^T \mathbf{r}_{-j}}{\|X_j\|^2}\right)^2 + \lambda|w_j| + \text{const}$$

This is exactly the proximal problem whose solution is the **soft-thresholding operator**
$\mathcal{S}_\lambda$ from `04.10`:

$$w_j \leftarrow \mathcal{S}_{\lambda/\|X_j\|^2}\!\left(\frac{X_j^T \mathbf{r}_{-j}}{\|X_j\|^2}\right)
= \text{sign}\!\left(\frac{X_j^T \mathbf{r}_{-j}}{\|X_j\|^2}\right)
\max\!\left(\left|\frac{X_j^T \mathbf{r}_{-j}}{\|X_j\|^2}\right| - \frac{\lambda}{\|X_j\|^2},\; 0\right)$$

Each coordinate update is $O(n)$ (one dot product), and a full sweep through all $d$
coordinates costs $O(nd)$ — the same as one gradient step. But CD's updates are *exact*
minimizations along each axis, not approximate gradient steps, so it typically needs far fewer
sweeps than ISTA needs gradient iterations. This is precisely what `sklearn.linear_model.Lasso`
and the celebrated GLMnet library (Friedman, Hastie & Tibshirani, 2010) compute.

### 3. Cyclic vs. randomized coordinate selection

**Cyclic CD** updates coordinates in a fixed order $j = 1, 2, \ldots, d, 1, 2, \ldots$ repeating.
This is deterministic, cache-friendly, and what GLMnet uses in practice. However, its
convergence analysis is surprisingly delicate — for general smooth convex functions, cyclic CD
can be slower than randomized CD on pathological examples (a carefully constructed 3×3
quadratic by Sun & Ye, 2019).

**Randomized CD** selects $j$ uniformly at random at each step. Its convergence theory is much
cleaner: for a smooth convex function with coordinate-wise Lipschitz constants $L_j$,
randomized CD satisfies (Nesterov, 2012):

$$\mathbb{E}[f(\mathbf{w}^{(t)}) - f^*] \leq \frac{2d \cdot \max_j L_j \cdot \|\mathbf{w}^{(0)} - \mathbf{w}^*\|^2}{t+4}$$

The $d$ factor reflects that each step updates only $1/d$ of the coordinates. Comparing to
full GD's rate $O(L R^2 / T)$: one full CD *sweep* ($d$ coordinate updates) costs the same as
one GD step, and the convergence per sweep is comparable — but CD handles the non-smooth
$\ell_1$ term for free (via soft-thresholding), which GD cannot.

In practice, cyclic CD often matches or beats randomized CD, despite the weaker theory.
The reason: real data matrices have structure (correlated columns, varying scales) that cyclic
ordering exploits implicitly. Most production solvers use cyclic CD with clever column ordering
(e.g., "greedy" CD picks the coordinate with the largest gradient magnitude — the Gauss-Southwell
rule — which provably achieves a rate depending on $\bar{L} = \frac{1}{d}\sum L_j$ instead of
$\max_j L_j$).

### 4. Block coordinate descent

Instead of updating a single coordinate, **block CD** partitions $\mathbf{w}$ into blocks
$\mathbf{w} = (\mathbf{w}_{B_1}, \ldots, \mathbf{w}_{B_K})$ and updates one block at a time:

$$\mathbf{w}_{B_k}^{(t+1)} = \arg\min_{\mathbf{w}_{B_k}} f(\mathbf{w}_{B_1}^{(t+1)}, \ldots, \mathbf{w}_{B_k}, \ldots, \mathbf{w}_{B_K}^{(t)})$$

This is essential for **group Lasso** $\sum_g \|\mathbf{w}_g\|_2$ where the regularizer is
separable across *groups* but not across individual coordinates within a group: each block
subproblem reduces to a group soft-thresholding operation
$\mathbf{w}_g \leftarrow (1 - \lambda/\|\mathbf{v}_g\|_2)_+ \mathbf{v}_g$ (shrink-and-kill
entire groups). Block CD also arises naturally in multi-task learning (one block per task),
matrix factorization (alternate between updating rows and columns — this is alternating
least squares), and training SVMs (SMO algorithm is block CD with blocks of size 2).

### 5. The convergence rate taxonomy

We have now encountered enough algorithms across `04.02`–`04.11` to build the complete
first-order convergence rate picture. The table below is the single most important reference
in this module — it tells you *how fast* each algorithm converges and *under what assumptions*:

| Algorithm | Assumption | Rate | Type | Notes |
| :--- | :--- | :--- | :--- | :--- |
| **Subgradient descent** | Convex, $G$-Lipschitz | $O(G/\sqrt{T})$ | Sublinear | Diminishing step $\eta_t = c/\sqrt{t}$ required; $f$ may *increase* per step |
| **Gradient descent** | Convex, $L$-smooth | $O(LR^2/T)$ | Sublinear | $R = \|\mathbf{w}_0 - \mathbf{w}^*\|$; halving error doubles iterations |
| **Nesterov accelerated GD** | Convex, $L$-smooth | $O(LR^2/T^2)$ | Sublinear (accel.) | Momentum-based; optimal among first-order methods |
| **GD (strongly convex)** | $\mu$-s.c., $L$-smooth | $O(\rho^T),\; \rho = 1 - \mu/L$ | Linear | Each step shrinks gap by fixed factor; rate = $1 - 1/\kappa$ |
| **Nesterov (strongly convex)** | $\mu$-s.c., $L$-smooth | $O(\rho^T),\; \rho = 1 - \sqrt{\mu/L}$ | Linear (accel.) | Best first-order rate; $\rho = 1 - 1/\sqrt{\kappa}$ |
| **ISTA** (`04.10`) | Composite ($f$ smooth + $g$ convex) | $O(1/T)$ | Sublinear | Proximal GD; same rate as plain GD |
| **FISTA** (`04.10`) | Composite ($f$ smooth + $g$ convex) | $O(1/T^2)$ | Sublinear (accel.) | Nesterov acceleration on proximal GD |
| **Coordinate descent** | Smooth + separable $g$ | $O(d/T)$ per sweep | Sublinear | Per-coordinate cost cheap; handles $\ell_1$ analytically |
| **Newton's method** (`04.07`) | Twice-diff., locally s.c. | Quadratic: $O(\rho^{2^T})$ | Superlinear | Needs Hessian ($O(d^3)$); local convergence only |

**Reading the table:** The "rate" column describes how the optimality gap
$f(\mathbf{w}_T) - f^*$ shrinks with iterations $T$. *Sublinear* means diminishing returns:
each additional iteration helps less than the previous one ($1/T$ means halving the error
requires doubling iterations). *Linear* means a constant fraction of the remaining error is
eliminated per step (exponential decay on a log scale — confusingly, "linear convergence"
looks linear on a *semilog* plot). *Quadratic/superlinear* means the number of correct digits
roughly doubles each step — Newton converges devastatingly fast once it's close, but only
locally and at $O(d^3)$ per-step cost.

The hierarchy from slow to fast: subgradient ($1/\sqrt{T}$) → GD/ISTA ($1/T$) → Nesterov/FISTA
($1/T^2$) → strongly-convex GD ($\rho^T$, $\rho = 1-1/\kappa$) → strongly-convex Nesterov
($\rho^T$, $\rho = 1-1/\sqrt{\kappa}$) → Newton (quadratic). Each jump requires either
stronger assumptions (smoothness, strong convexity) or more computation per step (Hessian).

### 6. Nesterov's lower bound and practical implications

Are these rates the best possible, or could a cleverer algorithm do better? Nesterov (2004)
proved a remarkable **lower bound**: for any first-order method (one that accesses $f$ only
through gradient/subgradient evaluations), there exist convex smooth functions for which:

- **Convex smooth:** $f(\mathbf{w}_T) - f^* \geq \Omega\!\left(\frac{LR^2}{T^2}\right)$ after $T$ steps
- **Strongly convex smooth:** $f(\mathbf{w}_T) - f^* \geq \Omega\!\left(\left(\frac{\sqrt{\kappa}-1}{\sqrt{\kappa}+1}\right)^{2T}\right)$

The worst-case construction is a **tridiagonal quadratic** $f(\mathbf{x}) = \frac{1}{2}\mathbf{x}^T A \mathbf{x}
- \mathbf{e}_1^T\mathbf{x}$ where $A$ has entries only on the main diagonal and its immediate
neighbors. The information-theoretic argument: after $T$ gradient evaluations, a first-order
method can only "explore" $T$ directions in $\mathbb{R}^d$ (the Krylov subspace
$\text{span}\{\nabla f(\mathbf{w}_0), \nabla f(\mathbf{w}_1), \ldots\}$), and on a tridiagonal
problem this means information propagates at most $T$ coordinates per step.

**Practical implications:**

1. **Plain GD is suboptimal.** GD achieves $O(1/T)$ for convex-smooth, but the lower bound is
   $\Omega(1/T^2)$. This is a factor-of-$T$ gap — GD leaves a *quadratic* amount of performance
   on the table. Nesterov's accelerated GD matches the lower bound and is therefore **optimal**.

2. **For strongly convex problems,** GD's rate is $1 - 1/\kappa$ while the optimal rate is
   $1 - 1/\sqrt{\kappa}$. Since $\sqrt{\kappa} \ll \kappa$ for ill-conditioned problems, Nesterov
   acceleration transforms, say, $\kappa = 10{,}000$ from needing $O(10{,}000)$ steps (GD) to
   $O(100)$ steps (Nesterov) — a $100\times$ speedup.

3. **Newton escapes these bounds** because it uses second-order information (the Hessian), not
   just gradients. The lower bound applies only to first-order methods — Newton's quadratic
   convergence is not a contradiction, it's a different (more expensive) information class.

4. **For practitioners:** the taxonomy tells you which optimizer to reach for. Sparse convex
   models (Lasso, elastic net) → coordinate descent or FISTA. Smooth convex, moderate $d$ →
   L-BFGS (`04.07`). Non-convex deep learning → Adam (`04.04`) with learning rate schedules
   (`04.06`). The convergence rate is not the whole story (per-step cost, memory, stochastic
   noise all matter), but it's the *theoretical* story, and understanding it prevents wasting
   time applying the wrong tool.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
import time
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Two panels below. **Left:** coordinate descent's characteristic zigzag path on a 2D
ill-conditioned quadratic — each step moves parallel to one axis (optimizing that coordinate
exactly), producing the staircase pattern that contrasts sharply with gradient descent's
diagonal trajectory. **Right:** the grand convergence rate comparison — all algorithms from
the taxonomy table racing on the same problem, with $f(\mathbf{w}_t) - f^*$ on a log scale.
The slopes directly reflect the theoretical rates: subgradient is flattest ($1/\sqrt{T}$),
Newton is steepest (quadratic), and the gap between GD and Nesterov demonstrates the
$O(1/T)$ vs $O(1/T^2)$ factor.

In [ ]:
# ── LEFT PANEL: CD zigzag on 2D contour ──
# ── RIGHT PANEL: Convergence rate comparison (all algorithms) ──

# --- Helper implementations for the convergence race ---
def soft_threshold(v, lam):
    """Soft-thresholding operator from 04.10."""
    return np.sign(v) * np.maximum(np.abs(v) - lam, 0.0)

def generate_lasso_problem(n, d, s, rng, noise=0.1):
    """Generate a sparse Lasso problem with s nonzero coefficients."""
    X = rng.normal(size=(n, d))
    w_true = np.zeros(d)
    w_true[:s] = rng.normal(size=s) * 3
    y = X @ w_true + noise * rng.normal(size=n)
    return X, y, w_true

def lasso_obj(w, X, y, lam):
    return 0.5 * np.sum((X @ w - y)**2) / len(y) + lam * np.sum(np.abs(w))

# --- 2D quadratic for the CD zigzag ---
mu_2d, L_2d = 1.0, 15.0
A_2d = np.diag([mu_2d, L_2d])

# CD path (cyclic): exact minimization along each coordinate of f(w) = 0.5 w^T A w
w_cd = np.array([4.0, 3.0])
cd_path = [w_cd.copy()]
for _ in range(20):
    for j in range(2):  # cyclic: j=0, j=1, j=0, ...
        # Exact 1D minimization: w_j* = 0 for diagonal quadratic centered at origin
        w_cd[j] = 0.0  # argmin_{w_j} 0.5 * A[j,j] * w_j^2 = 0
        cd_path.append(w_cd.copy())
cd_path = np.array(cd_path)

# GD path for comparison
w_gd = np.array([4.0, 3.0])
eta_gd = 2.0 / (mu_2d + L_2d)
gd_path = [w_gd.copy()]
for _ in range(40):
    w_gd = w_gd - eta_gd * (A_2d @ w_gd)
    gd_path.append(w_gd.copy())
gd_path = np.array(gd_path)

# --- Convergence race on a Lasso problem ---
rng = np.random.default_rng(42)
n, d, s = 200, 50, 8
X, y, w_true = generate_lasso_problem(n, d, s, rng)
lam = 0.05
L_lip = np.linalg.eigvalsh(X.T @ X / n).max()  # Lipschitz constant of smooth part
eta = 1.0 / L_lip
T_max = 300

# Find a very good solution as f* reference
w_ref = np.zeros(d)
for _ in range(5000):
    grad = X.T @ (X @ w_ref - y) / n
    w_ref = soft_threshold(w_ref - eta * grad, lam * eta)
f_star = lasso_obj(w_ref, X, y, lam)

# 1. Subgradient descent (use subgradient of |w|_1)
w = np.zeros(d)
gaps_subgrad = []
for t in range(1, T_max + 1):
    g = X.T @ (X @ w - y) / n + lam * np.sign(w)  # subgradient
    eta_t = 0.5 / np.sqrt(t)
    w = w - eta_t * g
    gaps_subgrad.append(lasso_obj(w, X, y, lam) - f_star)

# 2. Gradient descent (on the smooth part only — projected/proximal-free, treat as smooth approx)
# For fair comparison, use ISTA which IS the correct "GD" for composite problems

# 3. ISTA (proximal GD)
w = np.zeros(d)
gaps_ista = []
for t in range(T_max):
    grad = X.T @ (X @ w - y) / n
    w = soft_threshold(w - eta * grad, lam * eta)
    gaps_ista.append(lasso_obj(w, X, y, lam) - f_star)

# 4. FISTA (accelerated proximal GD)
w = np.zeros(d)
w_prev = w.copy()
gaps_fista = []
for t in range(1, T_max + 1):
    momentum = (t - 1) / (t + 2)
    v = w + momentum * (w - w_prev)
    grad = X.T @ (X @ v - y) / n
    w_new = soft_threshold(v - eta * grad, lam * eta)
    w_prev = w.copy()
    w = w_new
    gaps_fista.append(lasso_obj(w, X, y, lam) - f_star)

# 5. Coordinate descent (cyclic)
w = np.zeros(d)
col_norms_sq = np.sum(X**2, axis=0) / n
gaps_cd = []
for t in range(T_max):
    for j in range(d):
        r_j = y - X @ w + X[:, j] * w[j]  # residual without coordinate j
        rho_j = X[:, j].T @ r_j / n
        w[j] = soft_threshold(rho_j, lam) / col_norms_sq[j]
    gaps_cd.append(lasso_obj(w, X, y, lam) - f_star)

# 6. Newton-like (use full GD on a smooth strongly-convex quadratic for Newton demo)
# Newton on a quadratic converges in 1 step; use a general smooth function instead
# For the taxonomy plot, show the theoretical envelope rather than Newton on Lasso
# (Newton doesn't apply to non-smooth Lasso directly)

# --- PLOT ---
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

# Left: CD zigzag
ax = axes[0]
x1_grid = np.linspace(-5, 5, 200)
x2_grid = np.linspace(-4, 4, 200)
X1, X2 = np.meshgrid(x1_grid, x2_grid)
Z = 0.5 * (mu_2d * X1**2 + L_2d * X2**2)
ax.contour(X1, X2, Z, levels=20, cmap='Blues', alpha=0.7)
ax.plot(cd_path[:, 0], cd_path[:, 1], 'o-', color='#C44E52', lw=1.5, ms=3,
        label=f'Coordinate descent ({len(cd_path)-1} steps)', zorder=5)
ax.plot(gd_path[:, 0], gd_path[:, 1], 's-', color='#4C72B0', lw=1.5, ms=3,
        label=f'Gradient descent ({len(gd_path)-1} steps)', zorder=4)
ax.plot(0, 0, 'k*', ms=15, zorder=6)
ax.set_xlabel('$w_1$'); ax.set_ylabel('$w_2$')
ax.set_title(f'CD zigzag vs GD on ill-conditioned quadratic (κ={L_2d/mu_2d:.0f})')
ax.legend(fontsize=9); ax.set_aspect('equal')

# Right: Convergence rate race
ax = axes[1]
iters = np.arange(1, T_max + 1)
ax.semilogy(iters, np.maximum(gaps_subgrad, 1e-16), '-', color='#8C564B', lw=1.5,
            label='Subgradient $O(1/\\sqrt{T})$', alpha=0.8)
ax.semilogy(iters, np.maximum(gaps_ista, 1e-16), '-', color='#4C72B0', lw=1.5,
            label='ISTA $O(1/T)$', alpha=0.8)
ax.semilogy(iters, np.maximum(gaps_fista, 1e-16), '-', color='#55A868', lw=2,
            label='FISTA $O(1/T^2)$', alpha=0.8)
ax.semilogy(iters, np.maximum(gaps_cd, 1e-16), '-', color='#C44E52', lw=2,
            label='Coord. descent', alpha=0.8)

# Theoretical reference lines
c0 = gaps_ista[0]
ax.semilogy(iters, c0 * 5.0 / np.sqrt(iters), ':', color='#8C564B', alpha=0.4, label='_')
ax.semilogy(iters, c0 * 2.0 / iters, ':', color='#4C72B0', alpha=0.4, label='_')
ax.semilogy(iters, c0 * 2.0 / iters**2, ':', color='#55A868', alpha=0.4, label='_')

ax.set_xlabel('Iterations $T$'); ax.set_ylabel('$f(\mathbf{w}_T) - f^*$')
ax.set_title('Grand convergence rate comparison (Lasso problem)')
ax.legend(fontsize=9, loc='upper right')
ax.set_ylim(bottom=1e-14)

plt.tight_layout()
plt.show()

print(f"After {T_max} iterations:")
print(f"  Subgradient gap: {gaps_subgrad[-1]:.6e}")
print(f"  ISTA gap:        {gaps_ista[-1]:.6e}")
print(f"  FISTA gap:       {gaps_fista[-1]:.6e}")
print(f"  Coord. descent:  {gaps_cd[-1]:.6e}")
print(f"\nCD converges fastest here — each sweep does d={d} exact 1D minimizations,")
print(f"handling the L1 term analytically rather than via the proximal approximation.")

## 🐍 Implementation from Scratch

Three implementations below:

1. **Cyclic coordinate descent for Lasso** — the workhorse algorithm that sklearn and GLMnet
   use. Each sweep updates all $d$ coordinates via the closed-form soft-thresholding formula
   derived in the Theory section, maintaining a running residual for efficiency.

2. **Randomized coordinate descent** — same per-coordinate update, but with random selection.
   We compare convergence vs cyclic to see how the coordinate ordering affects practical speed.

3. **Grand convergence race** — all algorithms from the taxonomy (subgradient, ISTA, FISTA,
   cyclic CD, randomized CD) on a single Lasso problem, verifying the theoretical rate
   hierarchy empirically.

In [ ]:
# ── 1. Cyclic Coordinate Descent for Lasso ──
def cyclic_cd_lasso(X, y, lam, max_sweeps=200, tol=1e-10):
    """Cyclic coordinate descent for Lasso.
    Each sweep updates all d coordinates in order using the
    closed-form soft-thresholding update.
    Returns (w, history of objective values)."""
    n, d = X.shape
    w = np.zeros(d)
    residual = y.copy()  # r = y - Xw; maintained incrementally
    col_norms_sq = np.sum(X**2, axis=0) / n
    history = []
    for sweep in range(max_sweeps):
        for j in range(d):
            # Temporarily add back w_j's contribution to residual
            residual += X[:, j] * w[j]
            # Compute the unconstrained optimum for coordinate j
            rho_j = X[:, j] @ residual / n
            # Apply soft-thresholding
            w_j_new = soft_threshold(rho_j, lam) / col_norms_sq[j]
            # Update residual
            residual -= X[:, j] * w_j_new
            w[j] = w_j_new
        obj = 0.5 * np.sum(residual**2) / n + lam * np.sum(np.abs(w))
        history.append(obj)
        if sweep > 0 and abs(history[-2] - history[-1]) < tol:
            break
    return w, np.array(history)

# ── 2. Randomized Coordinate Descent for Lasso ──
def randomized_cd_lasso(X, y, lam, max_steps=None, rng=None):
    """Randomized coordinate descent for Lasso.
    Each step picks one coordinate uniformly at random.
    Records objective every d steps (one 'epoch') for fair comparison with cyclic CD."""
    n, d = X.shape
    if max_steps is None:
        max_steps = 200 * d  # 200 epochs
    if rng is None:
        rng = np.random.default_rng(0)
    w = np.zeros(d)
    residual = y.copy()
    col_norms_sq = np.sum(X**2, axis=0) / n
    history = []
    for t in range(max_steps):
        j = rng.integers(d)
        residual += X[:, j] * w[j]
        rho_j = X[:, j] @ residual / n
        w_j_new = soft_threshold(rho_j, lam) / col_norms_sq[j]
        residual -= X[:, j] * w_j_new
        w[j] = w_j_new
        if (t + 1) % d == 0:  # record every epoch
            obj = 0.5 * np.sum(residual**2) / n + lam * np.sum(np.abs(w))
            history.append(obj)
    return w, np.array(history)

# ── Test on the same Lasso problem ──
rng = np.random.default_rng(42)
n, d, s = 500, 200, 15
X, y, w_true = generate_lasso_problem(n, d, s, rng)
lam = 0.05
L_lip = np.linalg.eigvalsh(X.T @ X / n).max()
eta = 1.0 / L_lip

# Run cyclic CD
w_cyclic, hist_cyclic = cyclic_cd_lasso(X, y, lam, max_sweeps=150)

# Run randomized CD
w_rand, hist_rand = randomized_cd_lasso(X, y, lam, max_steps=150 * d, rng=np.random.default_rng(7))

# Run ISTA for comparison
w_ista = np.zeros(d)
hist_ista = []
for t in range(150):
    grad = X.T @ (X @ w_ista - y) / n
    w_ista = soft_threshold(w_ista - eta * grad, lam * eta)
    hist_ista.append(lasso_obj(w_ista, X, y, lam))
hist_ista = np.array(hist_ista)

# Run FISTA for comparison
w_fista = np.zeros(d)
w_fista_prev = w_fista.copy()
hist_fista = []
for t in range(1, 151):
    mom = (t - 1) / (t + 2)
    v = w_fista + mom * (w_fista - w_fista_prev)
    grad = X.T @ (X @ v - y) / n
    w_new = soft_threshold(v - eta * grad, lam * eta)
    w_fista_prev = w_fista.copy()
    w_fista = w_new
    hist_fista.append(lasso_obj(w_fista, X, y, lam))
hist_fista = np.array(hist_fista)

# Run subgradient descent
w_sub = np.zeros(d)
hist_sub = []
for t in range(1, 151):
    g = X.T @ (X @ w_sub - y) / n + lam * np.sign(w_sub)
    eta_t = 0.3 / np.sqrt(t)
    w_sub = w_sub - eta_t * g
    hist_sub.append(lasso_obj(w_sub, X, y, lam))
hist_sub = np.array(hist_sub)

# ── Grand convergence race plot ──
f_star = min(hist_cyclic[-1], hist_fista[-1])  # best achieved
sweeps = np.arange(1, 151)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: Cyclic vs Randomized CD
ax = axes[0]
n_plot = min(len(hist_cyclic), len(hist_rand))
ax.semilogy(sweeps[:n_plot], np.maximum(hist_cyclic[:n_plot] - f_star, 1e-16),
            '-', color='#C44E52', lw=2, label='Cyclic CD')
ax.semilogy(sweeps[:n_plot], np.maximum(hist_rand[:n_plot] - f_star, 1e-16),
            '--', color='#DD8452', lw=2, label='Randomized CD')
ax.set_xlabel('Sweeps (epochs)'); ax.set_ylabel('$f(\mathbf{w}) - f^*$')
ax.set_title('Cyclic vs Randomized Coordinate Descent')
ax.legend(fontsize=10)

# Right: Grand race — all algorithms
ax = axes[1]
ax.semilogy(sweeps, np.maximum(hist_sub - f_star, 1e-16), '-', color='#8C564B',
            lw=1.5, label='Subgradient', alpha=0.8)
ax.semilogy(sweeps, np.maximum(hist_ista - f_star, 1e-16), '-', color='#4C72B0',
            lw=1.5, label='ISTA', alpha=0.8)
ax.semilogy(sweeps, np.maximum(hist_fista - f_star, 1e-16), '-', color='#55A868',
            lw=2, label='FISTA', alpha=0.8)
ax.semilogy(sweeps[:len(hist_cyclic)], np.maximum(hist_cyclic - f_star, 1e-16),
            '-', color='#C44E52', lw=2, label='Cyclic CD', alpha=0.8)
ax.set_xlabel('Sweeps / Iterations'); ax.set_ylabel('$f(\mathbf{w}) - f^*$')
ax.set_title(f'Grand convergence race (n={n}, d={d}, s={s})')
ax.legend(fontsize=9)
ax.set_ylim(bottom=1e-15)

plt.tight_layout()
plt.show()

# Sparsity check
print(f"True nonzeros: {np.sum(np.abs(w_true) > 1e-8)}")
print(f"Cyclic CD nonzeros:     {np.sum(np.abs(w_cyclic) > 1e-8)}")
print(f"Randomized CD nonzeros: {np.sum(np.abs(w_rand) > 1e-8)}")
print(f"FISTA nonzeros:         {np.sum(np.abs(w_fista) > 1e-8)}")
print(f"\nFinal objectives (lower is better):")
print(f"  Subgradient:   {hist_sub[-1]:.8f}")
print(f"  ISTA:          {hist_ista[-1]:.8f}")
print(f"  FISTA:         {hist_fista[-1]:.8f}")
print(f"  Cyclic CD:     {hist_cyclic[-1]:.8f}")
print(f"  Randomized CD: {hist_rand[-1]:.8f}")

## 🤖 Why This Matters for AI

The convergence rate taxonomy is not just theory — it directly determines which optimizer
you'll find inside every major ML library:

**`sklearn.linear_model.Lasso` and `ElasticNet`** use **cyclic coordinate descent** internally
(the `cd` solver, which is the default). This is the algorithm we implemented above — the same
soft-thresholding update, looped over all coordinates. When you call `Lasso(alpha=0.1).fit(X, y)`,
sklearn is running exactly the cyclic CD loop from Cell 6.

**GLMnet** (Friedman, Hastie & Tibshirani, 2010) — the most widely cited sparse-model solver
in statistics — is coordinate descent with two critical enhancements: (1) **warm starts**
along the regularization path (solve at $\lambda_{\max}$, decrease $\lambda$ gradually, using
each solution as the starting point for the next — this makes the full path almost as cheap as
a single solve), and (2) **active-set strategies** (track which coordinates are nonzero and
only update those, periodically checking the others). Sklearn's `LassoCV` uses this path
strategy for cross-validation.

**`sklearn.linear_model.LogisticRegression(penalty='l1')`** also defaults to coordinate
descent (solver `liblinear` or `saga`). The coordinate-wise update for L1-regularized logistic
regression doesn't have a closed form (the smooth part isn't quadratic), so it uses a
one-dimensional Newton step per coordinate instead.

**When to use which optimizer — the practitioner's decision tree:**

| Problem type | Best optimizer | Why |
| :--- | :--- | :--- |
| Sparse convex (Lasso, elastic net, $d \gg n$) | **Coordinate descent** | Handles $\ell_1$ analytically; exploits sparsity; warm starts for paths |
| Sparse convex (moderate $d$, no path needed) | **FISTA** | Simpler to implement; $O(1/T^2)$ rate; no coordinate structure needed |
| Smooth convex, moderate $d$ ($< 10^4$) | **L-BFGS** (`04.07`) | Quasi-Newton; superlinear convergence; low memory |
| Non-convex deep learning | **Adam** (`04.04`) | Adaptive; handles stochastic gradients; robust to hyperparameters |
| Distributed / federated | **ADMM** (`04.12`) | Decomposes across machines; handles constraints naturally |

The convergence rates explain *why* this table looks the way it does: CD wins for sparse
problems because its per-coordinate cost is $O(\text{nnz})$ and it handles the non-smooth
$\ell_1$ penalty exactly; FISTA wins when you want a simple, general-purpose solver with
provably fast convergence; L-BFGS wins for smooth problems where second-order information
pays off; and Adam works for deep learning not because of any convergence guarantee (there
isn't one for non-convex problems) but because it's empirically robust to the wild
landscapes of `04.09`.

In [ ]:
# ── Timing benchmark: sklearn Lasso vs FISTA vs our CD ──
from sklearn.linear_model import Lasso

rng = np.random.default_rng(99)
n_bench, d_bench, s_bench = 1000, 5000, 20
X_bench, y_bench, w_bench_true = generate_lasso_problem(n_bench, d_bench, s_bench, rng)
lam_bench = 0.01

# sklearn Lasso (coordinate descent under the hood)
t0 = time.perf_counter()
model_sk = Lasso(alpha=lam_bench, fit_intercept=False, max_iter=500, tol=1e-6)
model_sk.fit(X_bench, y_bench)
t_sklearn = time.perf_counter() - t0
obj_sklearn = lasso_obj(model_sk.coef_, X_bench, y_bench, lam_bench)

# Our cyclic CD
t0 = time.perf_counter()
w_our_cd, hist_our_cd = cyclic_cd_lasso(X_bench, y_bench, lam_bench, max_sweeps=100)
t_our_cd = time.perf_counter() - t0
obj_our_cd = lasso_obj(w_our_cd, X_bench, y_bench, lam_bench)

# FISTA
L_bench = np.linalg.eigvalsh(X_bench.T @ X_bench / n_bench).max()
eta_bench = 1.0 / L_bench
t0 = time.perf_counter()
w_f = np.zeros(d_bench)
w_f_prev = w_f.copy()
for t in range(1, 301):
    mom = (t - 1) / (t + 2)
    v = w_f + mom * (w_f - w_f_prev)
    grad = X_bench.T @ (X_bench @ v - y_bench) / n_bench
    w_new = soft_threshold(v - eta_bench * grad, lam_bench * eta_bench)
    w_f_prev = w_f.copy()
    w_f = w_new
t_fista = time.perf_counter() - t0
obj_fista = lasso_obj(w_f, X_bench, y_bench, lam_bench)

print(f"Timing benchmark: n={n_bench}, d={d_bench}, s={s_bench}")
print(f"{'Method':20s} {'Time (s)':>10s} {'Objective':>14s} {'Nonzeros':>10s}")
print(f"{'─'*20} {'─'*10} {'─'*14} {'─'*10}")
print(f"{'sklearn Lasso':20s} {t_sklearn:10.4f} {obj_sklearn:14.8f} {np.sum(np.abs(model_sk.coef_) > 1e-8):10d}")
print(f"{'Our cyclic CD':20s} {t_our_cd:10.4f} {obj_our_cd:14.8f} {np.sum(np.abs(w_our_cd) > 1e-8):10d}")
print(f"{'FISTA (300 iters)':20s} {t_fista:10.4f} {obj_fista:14.8f} {np.sum(np.abs(w_f) > 1e-8):10d}")
print(f"\nsklearn is faster because its CD loop is in compiled Cython, but all three")
print(f"converge to the same objective. FISTA is slower here because each iteration")
print(f"requires a full O(nd) matrix-vector product, while CD's per-coordinate cost")
print(f"scales with column sparsity. For d >> n sparse problems, CD dominates.")

# Verify solutions agree
print(f"\nSolution agreement (max |w_sklearn - w_cd|): {np.max(np.abs(model_sk.coef_ - w_our_cd)):.2e}")
print(f"True nonzeros: {s_bench}, recovered by all methods: "
      f"{np.sum(np.abs(model_sk.coef_) > 1e-8)}, "
      f"{np.sum(np.abs(w_our_cd) > 1e-8)}, "
      f"{np.sum(np.abs(w_f) > 1e-8)}")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. Consider the 2D Lasso problem $\min_{w_1, w_2} \frac{1}{2}[(3w_1 + w_2 - 4)^2 + (w_1 + 2w_2 - 3)^2] + 0.5(|w_1| + |w_2|)$
   with $X = \begin{pmatrix}3 & 1\\1 & 2\end{pmatrix}$, $\mathbf{y} = (4, 3)^T$, $\lambda = 0.5$.
   Starting from $\mathbf{w}^{(0)} = (0, 0)$, execute one full cycle of coordinate descent
   (update $w_1$, then $w_2$) by hand. Show each soft-thresholding computation explicitly.

<details>
<summary>💡 Solution</summary>

Column norms: $\|X_1\|^2 = 9+1=10$, $\|X_2\|^2 = 1+4=5$. Starting residual:
$\mathbf{r} = \mathbf{y} - X\mathbf{w}^{(0)} = (4, 3)^T$.

**Update $w_1$:** $\rho_1 = X_1^T \mathbf{r} / \|X_1\|^2 = (3 \cdot 4 + 1 \cdot 3)/10 = 1.5$.
Soft-threshold: $\mathcal{S}_{0.5/10}(1.5) = \text{sign}(1.5)\max(|1.5| - 0.05, 0) = 1.45$.
So $w_1 = 1.45$. New residual: $\mathbf{r} = (4 - 3(1.45),\; 3 - 1(1.45))^T = (-0.35,\; 1.55)^T$.

**Update $w_2$:** $\rho_2 = X_2^T \mathbf{r} / \|X_2\|^2 = (1(-0.35) + 2(1.55))/5 = 0.55$.
Soft-threshold: $\mathcal{S}_{0.5/5}(0.55) = \text{sign}(0.55)\max(0.55 - 0.1, 0) = 0.45$.
So $w_2 = 0.45$.

After one sweep: $\mathbf{w}^{(1)} = (1.45, 0.45)$.

</details>

### 💭 UNDERSTAND
2. Coordinate descent works naturally for Lasso ($\lambda\|\mathbf{w}\|_1 = \lambda\sum_j |w_j|$)
   because the $\ell_1$ norm is **separable** — it decomposes into a sum of per-coordinate terms.
   Why does standard (single-coordinate) CD fail for the **group Lasso** penalty
   $\sum_g \|\mathbf{w}_g\|_2$? What modification is needed?

<details>
<summary>💡 Solution</summary>

The group Lasso penalty $\|\mathbf{w}_g\|_2 = \sqrt{\sum_{j \in g} w_j^2}$ is *not* separable
across individual coordinates within a group — changing $w_j$ affects $\|\mathbf{w}_g\|_2$
for every other coordinate in the same group. So the one-dimensional subproblem for a single
$w_j$ still involves the non-smooth $\ell_2$ group norm, which doesn't simplify to scalar
soft-thresholding.

The fix: use **block coordinate descent**, where each "coordinate" is an entire group
$\mathbf{w}_g$. The block subproblem *is* separable across groups (the penalty decomposes as
$\sum_g \|\mathbf{w}_g\|_2$), and each block update has a closed form:
$\mathbf{w}_g \leftarrow (1 - \lambda/\|\mathbf{v}_g\|_2)_+ \mathbf{v}_g$ — group
soft-thresholding, which either shrinks the entire group toward zero or kills it entirely
(setting all coordinates in the group to exactly zero).

</details>

### ✏️ DERIVE
3. Derive the coordinate-wise Lasso update from first principles: write the optimality condition
   $0 \in \partial_{w_j}\left[\frac{1}{2n}\|X\mathbf{w} - \mathbf{y}\|^2 + \lambda|w_j|\right]$,
   expand the subdifferential of $|w_j|$ into cases ($w_j > 0$, $w_j < 0$, $w_j = 0$), and
   solve each case to recover the soft-thresholding formula.

<details>
<summary>💡 Solution outline</summary>

The partial derivative of the smooth part with respect to $w_j$ is
$\frac{1}{n}X_j^T(X\mathbf{w} - \mathbf{y}) = \|X_j\|^2 w_j/n - X_j^T\mathbf{r}_{-j}/n$
where $\mathbf{r}_{-j}$ is the partial residual. Define $\rho_j = X_j^T \mathbf{r}_{-j}/n$
and $c_j = \|X_j\|^2/n$. The optimality condition is
$0 \in c_j w_j - \rho_j + \lambda \partial|w_j|$.

**Case 1** ($w_j > 0$): $\partial|w_j| = \{1\}$, so $w_j = (\rho_j - \lambda)/c_j$.
Valid only if $\rho_j > \lambda$.

**Case 2** ($w_j < 0$): $\partial|w_j| = \{-1\}$, so $w_j = (\rho_j + \lambda)/c_j$.
Valid only if $\rho_j < -\lambda$.

**Case 3** ($w_j = 0$): $\partial|w_j| = [-1, 1]$, so $0 \in -\rho_j + \lambda[-1,1]$,
i.e., $|\rho_j| \leq \lambda$.

Combining: $w_j = \mathcal{S}_{\lambda/c_j}(\rho_j/c_j) \cdot c_j / c_j
= \text{sign}(\rho_j)\max(|\rho_j| - \lambda, 0)/c_j$ — exactly the soft-thresholding update.

</details>

### 🐍 IMPLEMENT
4. Implement the **regularization path** for Lasso via coordinate descent with **warm starts**.
   Compute $\lambda_{\max} = \|X^T\mathbf{y}\|_\infty / n$ (the smallest $\lambda$ at which
   the solution is all zeros), create a grid of 50 $\lambda$ values from $\lambda_{\max}$ down
   to $0.01 \lambda_{\max}$, and solve each using coordinate descent, warm-starting from the
   previous solution. Plot the coefficient paths (each $w_j$ vs $\lambda$) and compare total
   time to solving each $\lambda$ cold-started from zero.

<details>
<summary>✅ How to know you're right</summary>

The warm-started path should be 3–10× faster than cold-started (each solve needs very few
sweeps because the previous solution is close). The coefficient paths should show the
characteristic Lasso pattern: coefficients enter the model one at a time as $\lambda$
decreases, with some crossing zero and changing sign. At $\lambda_{\max}$ all coefficients
are exactly zero; at $0.01\lambda_{\max}$ you should recover roughly the same number of
nonzeros as the true signal. Cross-check against `sklearn.linear_model.lasso_path`.

</details>

### 🤖 APPLY
5. Time `sklearn.Lasso` vs your FISTA implementation vs your cyclic coordinate descent on a
   problem with $n = 1000$, $d = 5000$, $s = 20$ nonzero coefficients. Which is fastest? Repeat
   with $d = 500$ (where $d < n$). Does the ranking change? Explain using the per-iteration
   cost analysis from the Theory section.

<details>
<summary>✅ What you should observe</summary>

For $d = 5000 \gg n$: sklearn Lasso is fastest (compiled Cython CD), your CD is second,
FISTA is slowest. CD wins over FISTA because each coordinate update is $O(n)$ and handles
the $\ell_1$ term exactly, while each FISTA step is $O(nd)$ for the full gradient.

For $d = 500 < n$: the gap narrows significantly, and FISTA may become competitive because
the $O(nd)$ gradient cost is smaller. L-BFGS on the smooth part (with proximal wrapping)
would likely win here — this is the regime where second-order information pays off.

</details>

### 🚀 CHALLENGE
6. Nesterov's lower bound $\Omega(1/T^2)$ implies GD's $O(1/T)$ rate is *suboptimal*. Construct
   the explicit worst-case function: a tridiagonal quadratic $f(\mathbf{x}) = \frac{1}{2}
   \mathbf{x}^T A \mathbf{x} - \mathbf{e}_1^T\mathbf{x}$ where $A$ is the $d \times d$ matrix
   with $A_{ii} = 2$, $A_{i,i+1} = A_{i+1,i} = -1$ (discretized Laplacian). Run GD and
   Nesterov's accelerated GD on this function for $d = 100$ and 500 iterations. Plot
   $f(\mathbf{w}_t) - f^*$ and verify that GD converges at exactly $\Theta(1/T)$ while
   Nesterov achieves $\Theta(1/T^2)$, matching the lower bound. Compute $\kappa(A)$ and
   verify it matches the theoretical $\kappa = O(d^2)$ for the discrete Laplacian.

<details>
<summary>🔍 What a strong answer covers</summary>

The tridiagonal matrix has eigenvalues $\lambda_k = 2 - 2\cos(k\pi/(d+1))$ for $k = 1,\ldots,d$,
giving $\mu = \lambda_1 \approx \pi^2/d^2$ and $L = \lambda_d \approx 4$, so $\kappa \approx
4d^2/\pi^2$. For $d = 100$, $\kappa \approx 4053$.

On a log-log plot, GD's gap should show slope $\approx -1$ (confirming $1/T$) and Nesterov's
should show slope $\approx -2$ (confirming $1/T^2$). This is the exact worst case: GD cannot
do better than $1/T$ on this function, and Nesterov matches the information-theoretic lower
bound. A strong answer notes that the information argument (Krylov subspace dimension = $T$
after $T$ steps, but the optimum requires exploring all $d$ coordinates of the tridiagonal
structure) is why this particular matrix is the hardest case.

</details>

---

## 📚 Further Reading
- Friedman, Hastie & Tibshirani, [*Regularization Paths for GLMs via Coordinate Descent*](https://www.jstatsoft.org/article/view/v033i01) (2010) — the GLMnet paper
- Nesterov, *Introductory Lectures on Convex Optimization: A Basic Course* (2004), Ch. 2 — lower bounds and optimal methods
- Bubeck, ["Convex Optimization: Algorithms and Complexity"](https://arxiv.org/abs/1405.4980) (2015) — the taxonomy in one survey
- Wright, ["Coordinate Descent Algorithms"](https://arxiv.org/abs/1502.04759) (2015) — comprehensive CD survey
- Beck & Teboulle, *A Fast Iterative Shrinkage-Thresholding Algorithm* (FISTA, 2009)

---

*Next notebook: [`04.12` — Advanced First-Order Methods](12_advanced_first_order_methods.ipynb)*